In [ ]:
import os
import calendar
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.spatial import KDTree
from datetime import date

In [ ]:
# ===== USER CONFIGURATION =====

# --- Paths ---
data_path  = os.path.expanduser('../test-data/v3.LR.historical_thkConc_ensembleStats')
mesh_file  = os.path.expanduser('../test-data/mpaso-IcoswISC30E3r5-restart.nc')
output_dir = os.path.expanduser('../figures')

# --- Route waypoint text file ---
# Format: one waypoint per line -> lat_deg, lon_deg, cumulative_dist_nm
# Lines beginning with '#' are treated as comments and ignored.
route_file_txt = os.path.expanduser('../test-data/text_route2.txt')
route_label    = os.path.splitext(os.path.basename(route_file_txt))[0]  # or set manually

# --- Years to include ---
year_start = 2025
year_end   = 2026
years = [str(y) for y in range(year_start, year_end + 1)]

# --- Output ---
save_figs = True   # set False to suppress file saving

# --- x-axis centering ---
# 0-based day-of-year to place at the centre of the heatmap x-axis.
# 258 ~ September 16, near the annual Arctic sea-ice minimum.
plot_center_doy = 258

# --- Spatial interpolation ---
# Number of nearest MPAS mesh cells to average over at each route waypoint
k_neighbors = 4

# --- Colormap limits ---
# Ice concentration is the ice area fraction (0–1).
# Ice volume/area is the grid-cell-mean effective thickness (m).
conc_vmin, conc_vmax = 0.0, 1.0
thk_vmin,  thk_vmax  = 0.0, 4.0

In [ ]:
print('Loading MPAS mesh...')
ds_mesh = xr.open_dataset(mesh_file)

rad2deg       = 180.0 / np.pi
lat_mesh_full = ds_mesh.latCell.values * rad2deg
lon_mesh_full = ds_mesh.lonCell.values * rad2deg

# Normalise longitudes to [-180, 180]
lon_mesh_full = np.where(lon_mesh_full > 180.0,
                         lon_mesh_full - 360.0,
                         lon_mesh_full)

# Restrict to cells at or north of 60 deg N
ind_north = np.where(lat_mesh_full >= 60.0)[0]
lat_mesh  = lat_mesh_full[ind_north]
lon_mesh  = lon_mesh_full[ind_north]

print(f'  Total mesh cells  : {len(lat_mesh_full):,}')
print(f'  Cells >= 60 deg N : {len(lat_mesh):,}')

In [ ]:
# ---- Load route waypoints from text file ----
print(f'Loading route from {route_file_txt} ...')

wp_lat_list  = []
wp_lon_list  = []
wp_dist_list = []

with open(route_file_txt) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = [p.strip() for p in line.split(',')]
        if len(parts) < 3:
            continue
        wp_lat_list.append(float(parts[0]))
        wp_lon_list.append(float(parts[1]))
        wp_dist_list.append(float(parts[2]))

trans_lat  = np.array(wp_lat_list)
trans_lon  = np.array(wp_lon_list)
cumul_dist = np.array(wp_dist_list)

seg_dist_nm   = np.diff(cumul_dist, prepend=0.0)
total_dist_nm = float(np.max(cumul_dist))

print(f'  Waypoints       : {len(trans_lat)}')
print(f'  Total distance  : {total_dist_nm:.1f} nm')
print(f'  Route label     : {route_label}')

# Build KDTree over the northern-hemisphere restricted mesh
print('Building KDTree...')
tree = KDTree(list(zip(lon_mesh, lat_mesh)))
_, inds = tree.query(list(zip(trans_lon, trans_lat)), k=k_neighbors)
print(f'  KDTree ready.  Neighbor index array shape: {inds.shape}')

In [ ]:
def _doy_index(year, month, day):
    """Return 0-based day-of-year index (0 = Jan 1)."""
    return (date(year, month, day) - date(year, 1, 1)).days


def _find_files(years, data_path):
    """Return sorted list of (year, month, filepath) tuples for available monthly files."""
    prefix  = 'v3.LR.historical_EnsStats.mpassi.hist.am.timeSeriesStatsDaily.'
    records = []
    for yr in years:
        for mo in range(1, 13):
            fname = os.path.join(data_path, f'{prefix}{yr}-{mo:02d}-01.nc')
            if os.path.exists(fname):
                records.append((int(yr), mo, fname))
    return records


# ---- Read and cache raw sea ice data ----
# raw_seaice_data holds a list of
# (year, month, conc_med, conc_5th, conc_95th, thk_med, thk_5th, thk_95th)
# where each array has shape (n_days_in_month, n_arctic_cells).

file_records = _find_files(years, data_path)

if not file_records:
    raise FileNotFoundError(
        f'No sea ice ensemble stats files found in {data_path} for years {years}.'
    )

raw_seaice_data = []
for yr, mo, fpath in file_records:
    print(f'  {os.path.basename(fpath)}')
    ds = xr.open_dataset(fpath)

    conc_med  = ds['timeDaily_avg_iceAreaCell_ensembleMedian'].values
    conc_5th  = ds['timeDaily_avg_iceAreaCell_ensemble5th'].values
    conc_95th = ds['timeDaily_avg_iceAreaCell_ensemble95th'].values
    thk_med   = ds['timeDaily_avg_iceVolumeCell_ensembleMedian'].values
    thk_5th   = ds['timeDaily_avg_iceVolumeCell_ensemble5th'].values
    thk_95th  = ds['timeDaily_avg_iceVolumeCell_ensemble95th'].values
    ds.close()

    raw_seaice_data.append((yr, mo,
                            conc_med, conc_5th, conc_95th,
                            thk_med,  thk_5th,  thk_95th))

print(f'\nRaw data read complete. {len(raw_seaice_data)} file(s) cached.')

In [ ]:
# ---- Route-specific extraction ----
# For each day, average k nearest-neighbor cell values at each waypoint,
# then take the route-maximum (highest ice = worst-case conditions).

avail_years  = sorted(set(yr for yr, *_ in raw_seaice_data))
n_years      = len(avail_years)
year_idx_map = {yr: i for i, yr in enumerate(avail_years)}

shape = (n_years, 366)

max_conc_med  = np.full(shape, np.nan)
max_conc_5th  = np.full(shape, np.nan)
max_conc_95th = np.full(shape, np.nan)
max_thk_med   = np.full(shape, np.nan)
max_thk_5th   = np.full(shape, np.nan)
max_thk_95th  = np.full(shape, np.nan)
max_prod_med  = np.full(shape, np.nan)
max_prod_5th  = np.full(shape, np.nan)
max_prod_95th = np.full(shape, np.nan)

for (yr, mo,
     conc_med_arr, conc_5th_arr, conc_95th_arr,
     thk_med_arr,  thk_5th_arr,  thk_95th_arr) in raw_seaice_data:

    n_days = conc_med_arr.shape[0]
    yi     = year_idx_map[yr]

    for day_idx in range(n_days):
        doy = _doy_index(yr, mo, day_idx + 1)   # 0-based day-of-year

        # Average k neighbours at each route waypoint: shape (n_waypoints,)
        wp_conc_med  = np.mean(conc_med_arr [day_idx, inds], axis=1)
        wp_conc_5th  = np.mean(conc_5th_arr [day_idx, inds], axis=1)
        wp_conc_95th = np.mean(conc_95th_arr[day_idx, inds], axis=1)
        wp_thk_med   = np.mean(thk_med_arr  [day_idx, inds], axis=1)
        wp_thk_5th   = np.mean(thk_5th_arr  [day_idx, inds], axis=1)
        wp_thk_95th  = np.mean(thk_95th_arr [day_idx, inds], axis=1)

        # Route-maximum: highest ice along the route (worst-case condition)
        max_conc_med [yi, doy] = float(np.nanmax(wp_conc_med))
        max_conc_5th [yi, doy] = float(np.nanmax(wp_conc_5th))
        max_conc_95th[yi, doy] = float(np.nanmax(wp_conc_95th))
        max_thk_med  [yi, doy] = float(np.nanmax(wp_thk_med))
        max_thk_5th  [yi, doy] = float(np.nanmax(wp_thk_5th))
        max_thk_95th [yi, doy] = float(np.nanmax(wp_thk_95th))
        max_prod_med [yi, doy] = float(np.nanmax(wp_conc_med  * wp_thk_med))
        max_prod_5th [yi, doy] = float(np.nanmax(wp_conc_5th  * wp_thk_5th))
        max_prod_95th[yi, doy] = float(np.nanmax(wp_conc_95th * wp_thk_95th))

print('Route extraction complete.')
print(f'  {n_years} year(s), {len(raw_seaice_data)} file(s) processed.')

In [ ]:
# Month start positions (0-based day-of-year) and labels for x-axis ticks
_MONTH_DOY   = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
_MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


def make_seaice_heatmap(data, years, field_label, stat_label,
                        route_label, save_figs, output_dir,
                        vmin, vmax, cmap_name,
                        cbar_label, field_slug,
                        center_doy=258):
    """
    Produce a single sea-ice heatmap figure.

    Parameters
    ----------
    data        : ndarray, shape (n_years, 366)
                  Route-maximum sea ice value for each year/day.
    years       : list of int
    field_label : str   e.g. 'Ice Concentration', 'Ice Thickness'
    stat_label  : str   e.g. 'Median', '5th Percentile'
    route_label : str
    save_figs   : bool
    output_dir  : str
    vmin, vmax  : float  colorbar limits
    cmap_name   : str    matplotlib colormap name
    cbar_label  : str    colorbar axis label
    field_slug  : str    short identifier for filenames (e.g. 'conc', 'thk')
    center_doy  : int    0-based DOY to place at the centre of the x-axis
    """
    n_years  = len(years)
    n_days   = 365         # omit leap-year slot at index 365
    plot_arr = data[:, :n_days].copy()

    # Roll columns so that center_doy sits in the middle of the x-axis
    start_doy = (center_doy - n_days // 2) % n_days
    plot_arr  = np.roll(plot_arr, -start_doy, axis=1)

    # Month positions in the rolled frame
    shifted_starts = [(_MONTH_DOY[m] - start_doy) % n_days for m in range(12)]
    month_order    = sorted(range(12), key=lambda m: shifted_starts[m])

    # Derive vmax from the data when not explicitly provided
    if vmax is None:
        vmax = float(np.nanmax(plot_arr)) if np.any(np.isfinite(plot_arr)) else 1.0

    fig_height = max(3.5, n_years * 0.50 + 2.0)
    fig, ax = plt.subplots(figsize=(16, fig_height))

    extent = [-0.5, n_days - 0.5, -0.5, n_years - 0.5]

    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad(color='lightgrey')
    masked = np.ma.masked_invalid(plot_arr)

    im = ax.imshow(masked, aspect='auto', origin='lower',
                   cmap=cmap, vmin=vmin, vmax=vmax,
                   extent=extent, interpolation='none')

    cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
    cbar.set_label(cbar_label, fontsize=11)

    # x-axis: month tick marks and vertical grid lines
    tick_positions = [shifted_starts[m] for m in month_order]
    tick_labels    = [_MONTH_NAMES[m] for m in month_order]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=10)
    ax.set_xlim(-0.5, n_days - 0.5)

    for m in range(12):
        pos = shifted_starts[m]
        if 0 < pos < n_days:
            ax.axvline(x=pos, color='white', linewidth=0.6, alpha=0.5)

    # y-axis: one tick per year, earliest year at the bottom
    ax.set_yticks(range(n_years))
    ax.set_yticklabels([str(y) for y in years], fontsize=10)
    ax.set_ylabel('Year', fontsize=11)
    ax.set_xlabel('Month', fontsize=11)

    ax.set_title(
        f'{field_label}  —  {stat_label}  |  Route {route_label}',
        fontsize=12, fontweight='bold'
    )

    plt.tight_layout()

    if save_figs:
        os.makedirs(output_dir, exist_ok=True)
        stat_slug = (stat_label.lower()
                     .replace(' ', '_')
                     .replace('th', '')
                     .replace('st', ''))
        fname = os.path.join(
            output_dir,
            f'seaice_{field_slug}_route{route_label}_{stat_slug}.png'
        )
        fig.savefig(fname, dpi=150, bbox_inches='tight')

        print(f'  Saved: {fname}')    

    plt.show()
    plt.close(fig)


In [ ]:
# ---- Generate plots ----
# Three fields × three ensemble statistics = nine heatmaps.

# (data_array, stat_label, field_label, cmap_name, vmin, vmax, cbar_label, field_slug)
plot_specs = [
    # --- Ice Concentration ---
    (max_conc_5th,  '5th Percentile',  'Ice Concentration (route-max)',
     'Blues_r', conc_vmin, conc_vmax,
     'Ice area fraction (route-maximum)', 'conc'),
    (max_conc_med,  'Median',           'Ice Concentration (route-max)',
     'Blues_r', conc_vmin, conc_vmax,
     'Ice area fraction (route-maximum)', 'conc'),
    (max_conc_95th, '95th Percentile',  'Ice Concentration (route-max)',
     'Blues_r', conc_vmin, conc_vmax,
     'Ice area fraction (route-maximum)', 'conc'),
    # --- Ice Thickness (vmax=None -> derived from each plot's data) ---
    (max_thk_5th,  '5th Percentile',   'Ice Thickness (route-max)',
     'Blues_r', thk_vmin, None,
     'Effective ice thickness (m, route-maximum)', 'thk'),
    (max_thk_med,  'Median',            'Ice Thickness (route-max)',
     'Blues_r', thk_vmin, None,
     'Effective ice thickness (m, route-maximum)', 'thk'),
    (max_thk_95th, '95th Percentile',   'Ice Thickness (route-max)',
     'Blues_r', thk_vmin, None,
     'Effective ice thickness (m, route-maximum)', 'thk'),
    # --- Concentration × Thickness product (vmax=None -> derived from each plot's data) ---
    (max_prod_5th,  '5th Percentile',  'Conc \u00d7 Thickness (route-max)',
     'Blues_r', 0.0, None,
     'Conc \u00d7 Thickness (m, route-maximum)', 'prod'),
    (max_prod_med,  'Median',           'Conc \u00d7 Thickness (route-max)',
     'Blues_r', 0.0, None,
     'Conc \u00d7 Thickness (m, route-maximum)', 'prod'),
    (max_prod_95th, '95th Percentile',  'Conc \u00d7 Thickness (route-max)',
     'Blues_r', 0.0, None,
     'Conc \u00d7 Thickness (m, route-maximum)', 'prod'),
]

for (data, stat_label, field_label, cmap_name,
     vmin, vmax, cbar_label, field_slug) in plot_specs:
    print(f'Plotting {field_label}  —  {stat_label} ...')
    make_seaice_heatmap(
        data        = data,
        years       = avail_years,
        field_label = field_label,
        stat_label  = stat_label,
        route_label = route_label,
        save_figs   = save_figs,
        output_dir  = output_dir,
        vmin        = vmin,
        vmax        = vmax,
        cmap_name   = cmap_name,
        cbar_label  = cbar_label,
        field_slug  = field_slug,
        center_doy  = plot_center_doy,
    )